In [15]:
import json
import pandas as pd
import ast

In [16]:
df = pd.read_csv('bitcoin_tweet_2022.csv')
df

,date,text2,hashtags
0,2022-01-14,death cross bitcoin dump,['bitcoin']
1,2022-01-14,teaser bitcoin cryptocurrency furniture,"['bitcoin', 'cryptocurrency', 'furniture']"
2,2022-01-14,well time bitcoin mocked made fun upon careful,['bitcoin']
3,2022-01-14,bitcoin christmas ethereum shibthis opportunit...,"['bitcoin', 'christmas', 'ethereum', 'shib', '..."
4,2022-01-14,ainu token coming soon play earntelegram ainu ...,"['ainutoken', 'playtoearn', 'nfts', 'metaverse..."
...,...,...,...
839594,2022-12-24,making noise crypto bsc ethereum bitcoin alt c...,"['crypto', 'bsc', 'ethereum', 'bitcoin', 'alt']"
839595,2022-12-24,russiaukraine war reason biggest bitcoin sello...,['bitcoin']
839596,2022-12-24,glassnodealerts bitcoin btc percent supply las...,['bitcoin']
839597,2022-12-24,dont consider bitcoin worth much doesnt meet s...,"['bitcoin', 'decred', 'btc', 'dcr']"


In [17]:
df['hashtags'] = df['hashtags'].str.lower()
df['hashtags'] = df['hashtags'].apply(ast.literal_eval)

In [19]:
df_expanded = df.explode('hashtags')
df_expanded

,date,text2,hashtags
0,2022-01-14,death cross bitcoin dump,bitcoin
1,2022-01-14,teaser bitcoin cryptocurrency furniture,bitcoin
1,2022-01-14,teaser bitcoin cryptocurrency furniture,cryptocurrency
1,2022-01-14,teaser bitcoin cryptocurrency furniture,furniture
2,2022-01-14,well time bitcoin mocked made fun upon careful,bitcoin
...,...,...,...
839597,2022-12-24,dont consider bitcoin worth much doesnt meet s...,bitcoin
839597,2022-12-24,dont consider bitcoin worth much doesnt meet s...,decred
839597,2022-12-24,dont consider bitcoin worth much doesnt meet s...,btc
839597,2022-12-24,dont consider bitcoin worth much doesnt meet s...,dcr


In [61]:
threshold = 8500
hashtag_counts = df_expanded['hashtags'].value_counts()
valid_hashtags = hashtag_counts[hashtag_counts >= threshold].index

df_filter = df_expanded[df_expanded['hashtags'].isin(valid_hashtags)]
df_filter = df_filter[['date','hashtags']]
print(df_filter['hashtags'].value_counts())
len(df_filter['hashtags'].unique())

hashtags
bitcoin               594814
btc                   433007
crypto                192940
eth                   137202
cryptocurrency        120272
ethereum               86216
nft                    82751
binance                57671
blockchain             53817
bnb                    51760
nfts                   48395
cryptonews             42199
bsc                    34626
defi                   34085
metaverse              33819
trading                30327
nftcommunity           30058
altcoin                28148
doge                   21459
dogecoin               19221
shib                   18563
xrp                    18548
solana                 18478
cryptocurrencies       18013
airdrop                17489
nftart                 16254
ada                    15710
web3                   15233
altcoins               14491
cryptocrash            12828
cryptotrading          12797
luna                   11876
cryptomining           11464
sol                    10855
usdt 

50

In [62]:
df_filter['date'] = pd.to_datetime(df_filter['date'])
df_filter

,date,hashtags
0,2022-01-14,bitcoin
1,2022-01-14,bitcoin
1,2022-01-14,cryptocurrency
2,2022-01-14,bitcoin
3,2022-01-14,bitcoin
...,...,...
839595,2022-12-24,bitcoin
839596,2022-12-24,bitcoin
839597,2022-12-24,bitcoin
839597,2022-12-24,btc


In [63]:
from tqdm import tqdm

full_dates = pd.date_range(start=df_filter['date'].min(), end=df_filter['date'].max())
hashtags = df_filter['hashtags'].unique()
result = []

for tag in tqdm(hashtags, desc="Processing hashtags"):
    df_tag = df_filter[df_filter['hashtags'] == tag]
    daily_counts = df_tag.groupby('date').size()
    daily_counts = daily_counts.reindex(full_dates, fill_value=0)

    timeline = [
        {"date": date.strftime('%Y-%m-%d'), "count": int(count)}
        for date, count in daily_counts.items()
    ]

    result.append({
        "hashtag": tag,
        "timeline": timeline
    })


Processing hashtags: 100%|██████████| 50/50 [00:09<00:00,  5.06it/s]


In [64]:

with open("hashtag_timeline.json", "w") as f:
    json.dump(result, f, indent=4)